# 07. Public dataset: YOLOv8n-HBB

Trains and evaluates **one** model with the common protocol used for all 12 models of the thesis:
same library version (Ultralytics 8.4.175), same cleaned data for HBB and OBB, same hyper-parameters (100 epochs, patience 20, batch 16, 640 px, AdamW 0.002, seed set explicitly), detection metrics on the **test** split, per-image counting at several confidence thresholds (MAE, MAPE, RMSE, bias), and a repeated FPS measurement.

**Before running:** add the two Kaggle datasets as input, turn on the GPU, and run all cells. Download the files in `/kaggle/working/results/` when finished.

In [14]:
# Same library version for ALL 12 notebooks (the old runs used 8.4.163-8.4.175, which changed the head and the pretrained-weight transfer).
!pip install -q ultralytics==8.4.175

In [15]:
# ======================= EDIT ONLY THIS CELL =======================
DATASET  = "public"     # "custom" (Peshawar UAV) or "public" (Roboflow solar panel)
MODEL    = "yolov8n"       # "yolov8n" | "yolo11n" | "yolo26n"
BOX      = "HBB"         # "HBB" (horizontal boxes) or "OBB" (oriented boxes)
SEED     = 0               # change to 1, 2 to repeat the run with another seed
OBB_INIT = "coco"          # OBB only. "coco": start from the COCO *detection* weights (same start as HBB = controlled comparison)
                           #               "dota": start from the official Ultralytics OBB weights (pretrained on DOTA)

# Dataset exports (the SAME files are used for HBB and OBB and for all three YOLO versions).
# Add both Kaggle datasets to the notebook ("Add Input") and check these paths.
SRC = {
    "public": "/kaggle/input/datasets/hamzazafar12/publicdataset-yolo-v8-hbb/solar panel.v2i.yolov8",
}

# Identical for every model:
EPOCHS, PATIENCE, BATCH, IMGSZ = 100, 20, 16, 640
CONF_PRIMARY = 0.25                      # confidence threshold used for the main counting results
THRESHOLDS   = [0.10, 0.25, 0.40, 0.50]  # counts are also reported at these thresholds
IOU_NMS, MAX_DET = 0.7, 1000             # NMS IoU and maximum detections per image (the old runs used 300)

RUN_NAME = f"{DATASET}_{MODEL}_{BOX}_seed{SEED}"
WORK, RESULTS_DIR, RUNS_DIR = "/kaggle/working", "/kaggle/working/results", "/kaggle/working/runs"

### 1. Helpers

In [16]:
import os, shutil, csv, json, math, statistics, time, platform
from pathlib import Path
import numpy as np
import yaml, torch
import ultralytics
from ultralytics import YOLO

EXTS = (".jpg", ".jpeg", ".png")
SPLIT_NAMES = {"train": ["train"], "val": ["valid", "val"], "test": ["test"]}

def find_split(src, names):
    for n in names:
        if os.path.isdir(os.path.join(src, n, "images")):
            return n
    raise FileNotFoundError(f"No images folder found in {src} for {names}")

def poly_area(pts):
    s = 0.0
    for i in range(len(pts)):
        x1, y1 = pts[i]; x2, y2 = pts[(i + 1) % len(pts)]
        s += x1 * y2 - x2 * y1
    return abs(s) / 2.0

def clean_rows(label_path):
    """Read a YOLO label file and return clean polygon rows for class 0.
    - 4-value rows (x y w h) become rectangle polygons, so every row is a polygon
    - coordinates are clipped to [0, 1]
    - degenerate polygons (fewer than 3 points or ~zero area) are dropped
    """
    rows, st = [], dict(rows=0, box_rows=0, poly_rows=0, dropped=0, clipped=0)
    if not os.path.exists(label_path):
        return rows, st
    with open(label_path, encoding="utf-8") as f:
        for ln in f:
            parts = ln.split()
            if not parts:
                continue
            st["rows"] += 1
            try:
                vals = [float(v) for v in parts[1:]]
            except ValueError:
                st["dropped"] += 1; continue
            if len(vals) == 4:
                x, y, w, h = vals
                pts = [(x - w / 2, y - h / 2), (x + w / 2, y - h / 2), (x + w / 2, y + h / 2), (x - w / 2, y + h / 2)]
                st["box_rows"] += 1
            elif len(vals) >= 6 and len(vals) % 2 == 0:
                pts = list(zip(vals[0::2], vals[1::2]))
                st["poly_rows"] += 1
            else:
                st["dropped"] += 1; continue
            if any(px < 0 or px > 1 or py < 0 or py > 1 for px, py in pts):
                st["clipped"] += 1
            pts = [(min(max(px, 0.0), 1.0), min(max(py, 0.0), 1.0)) for px, py in pts]
            if poly_area(pts) < 1e-8:
                st["dropped"] += 1; continue
            rows.append("0 " + " ".join(f"{px:.6f} {py:.6f}" for px, py in pts))
    return rows, st

def prepare_dataset(src, dst):
    """Copy a Roboflow YOLO export to a writable folder with cleaned labels; return data.yaml path and a report."""
    if os.path.exists(dst):
        shutil.rmtree(dst)
    report = {}
    for split, names in SPLIT_NAMES.items():
        s = find_split(src, names)
        os.makedirs(f"{dst}/{split}/images"); os.makedirs(f"{dst}/{split}/labels")
        tot = dict(images=0, rows=0, box_rows=0, poly_rows=0, dropped=0, clipped=0, kept=0, max_per_image=0)
        for fn in sorted(os.listdir(os.path.join(src, s, "images"))):
            if not fn.lower().endswith(EXTS):
                continue
            shutil.copy2(os.path.join(src, s, "images", fn), f"{dst}/{split}/images/{fn}")
            rows, st = clean_rows(os.path.join(src, s, "labels", os.path.splitext(fn)[0] + ".txt"))
            with open(f"{dst}/{split}/labels/{os.path.splitext(fn)[0]}.txt", "w", encoding="utf-8") as f:
                f.write("\n".join(rows) + ("\n" if rows else ""))
            tot["images"] += 1; tot["kept"] += len(rows); tot["max_per_image"] = max(tot["max_per_image"], len(rows))
            for k in ("rows", "box_rows", "poly_rows", "dropped", "clipped"):
                tot[k] += st[k]
        report[split] = tot
    data_yaml = f"{dst}/data.yaml"
    with open(data_yaml, "w", encoding="utf-8") as f:
        yaml.safe_dump({"path": dst, "train": "train/images", "val": "val/images", "test": "test/images",
                        "nc": 1, "names": {0: "solar panel"}}, f)
    return data_yaml, report

def gt_count(label_file):
    if not os.path.exists(label_file):
        return 0
    with open(label_file, encoding="utf-8") as f:
        return sum(1 for ln in f if ln.strip())

def counting_metrics(gt, pred):
    n = len(gt)
    err = [p - g for g, p in zip(gt, pred)]
    ae = [abs(e) for e in err]
    rel = [abs(p - g) / g for g, p in zip(gt, pred) if g > 0]
    return {"MAE": sum(ae) / n,
            "RMSE": math.sqrt(sum(e * e for e in err) / n),
            "MAPE_pct": 100 * sum(rel) / len(rel) if rel else float("nan"),
            "bias": sum(err) / n,
            "median_AE": statistics.median(ae),
            "max_AE": max(ae)}

def f1_score(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

### 2. Prepare the data (identical files for every model of this dataset)

In [17]:
assert torch.cuda.is_available(), "No GPU found: Notebook settings -> Accelerator -> GPU (T4 x2 or P100)."
print("GPU:", torch.cuda.get_device_name(0), "| Ultralytics", ultralytics.__version__)
assert os.path.isdir(SRC[DATASET]), f"Dataset folder not found: {SRC[DATASET]}\nAvailable under /kaggle/input: {os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'n/a'}"
LOCAL = f"{WORK}/data_{DATASET}"
DATA_YAML, report = prepare_dataset(SRC[DATASET], LOCAL)
print("Cleaned dataset written to", LOCAL)
for split, r in report.items():
    print(f"{split:5s} images={r['images']:4d} labels kept={r['kept']:6d} (read {r['rows']}, box rows converted {r['box_rows']}, "
          f"dropped {r['dropped']}, clipped {r['clipped']}) max per image={r['max_per_image']}")

GPU: Tesla T4 | Ultralytics 8.4.175
Cleaned dataset written to /kaggle/working/data_public
train images= 510 labels kept=  4511 (read 4511, box rows converted 0, dropped 0, clipped 0) max per image=109
val   images= 109 labels kept=  1052 (read 1052, box rows converted 1, dropped 0, clipped 0) max per image=149
test  images= 109 labels kept=  1332 (read 1332, box rows converted 0, dropped 0, clipped 0) max per image=56


### 3. Train

In [18]:
INIT = f"{MODEL}.pt"                       # official COCO detection weights
model = YOLO(INIT)

print("Initial weights:", INIT)
model.train(data=DATA_YAML, epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
            seed=SEED, deterministic=True, optimizer="AdamW", lr0=0.002, momentum=0.9,
            device=0, workers=2, project=RUNS_DIR, name=RUN_NAME, exist_ok=True)
SAVE_DIR = Path(model.trainer.save_dir)
BEST = SAVE_DIR / "weights" / "best.pt"
EPOCHS_RUN = int(model.trainer.epoch) + 1
print("Best weights:", BEST, "| epochs run:", EPOCHS_RUN)

Initial weights: yolov8n.pt
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_public/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, 

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      3.42G      1.136      1.291      1.251        261        640: 100% ━━━━━━━━━━━━ 32/32 6.0it/s 5.4s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.1it/s 1.0s0.4s
                   all        109       1052      0.549      0.426      0.412      0.281

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      3.43G      1.095      1.258      1.226        193        640: 100% ━━━━━━━━━━━━ 32/32 5.4it/s 5.9s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.3it/s 0.9s0.4s
                   all        109       1052      0.577       0.39      0.407       0.27

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      3.43G      1.043        1.1      1.204        234        640: 100% ━━━━━━━━━━━━ 32/32 5.8it/s 5.5s0.5s
                 Class     Images  Instances  

### 4. Detection metrics on the validation and test splits

In [19]:
ev = YOLO(str(BEST))
IS_OBB = (BOX == "OBB")

def det_metrics(split):
    m = ev.val(data=DATA_YAML, split=split, imgsz=IMGSZ, batch=BATCH, conf=0.001, iou=IOU_NMS,
               max_det=MAX_DET, plots=False, verbose=False)
    P, R = float(m.box.mp), float(m.box.mr)
    return {"P": P, "R": R, "F1": f1_score(P, R), "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map)}

val_m, test_m = det_metrics("val"), det_metrics("test")
print("VALIDATION split:", {k: round(v, 4) for k, v in val_m.items()})
print("TEST split      :", {k: round(v, 4) for k, v in test_m.items()})

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1547.7±455.5 MB/s, size: 59.1 KB)
val: Scanning /kaggle/working/data_public/val/labels.cache... 109 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 109/109 32.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 6.5it/s 1.1s0.1s
                   all        109       1052      0.929      0.849      0.903      0.766
Speed: 0.8ms preprocess, 4.4ms inference, 0.0ms loss, 0.9ms postprocess per image
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1509.6±670.5 MB/s, size: 62.2 KB)
val: Scanning /kaggle/working/data_public/test/labels... 109 im

### 5. Counting on the test images

In [20]:
img_dir, lbl_dir = f"{LOCAL}/test/images", f"{LOCAL}/test/labels"
imgs = sorted(f for f in os.listdir(img_dir) if f.lower().endswith(EXTS))
low = min(THRESHOLDS)
per_image, gts, preds = [], [], {t: [] for t in THRESHOLDS}
for im in imgs:
    r = ev.predict(os.path.join(img_dir, im), conf=low - 1e-6, iou=IOU_NMS, max_det=MAX_DET,
                   imgsz=IMGSZ, verbose=False, device=0)[0]
    det = r.obb if IS_OBB else r.boxes
    confs = det.conf.cpu().numpy() if (det is not None and len(det)) else np.array([])
    g = gt_count(os.path.join(lbl_dir, os.path.splitext(im)[0] + ".txt"))
    gts.append(g)
    row = {"image": im, "gt_count": g}
    for t in THRESHOLDS:
        c = int((confs > t).sum()); preds[t].append(c); row[f"pred@{t}"] = c
    per_image.append(row)

count_res = {t: counting_metrics(gts, preds[t]) for t in THRESHOLDS}
print(f"{len(imgs)} test images, max ground-truth count = {max(gts)}")
for t in THRESHOLDS:
    c = count_res[t]
    print(f"conf>{t:.2f}: MAE={c['MAE']:.2f}  MAPE={c['MAPE_pct']:.2f}%  RMSE={c['RMSE']:.2f}  bias={c['bias']:+.2f}  median|e|={c['median_AE']:.1f}  max|e|={c['max_AE']:.0f}")

109 test images, max ground-truth count = 56
conf>0.10: MAE=2.86  MAPE=26.16%  RMSE=5.13  bias=+1.38  median|e|=1.0  max|e|=24
conf>0.25: MAE=2.00  MAPE=15.87%  RMSE=4.59  bias=-0.26  median|e|=0.0  max|e|=27
conf>0.40: MAE=1.96  MAPE=14.91%  RMSE=4.88  bias=-0.99  median|e|=0.0  max|e|=28
conf>0.50: MAE=2.04  MAPE=14.65%  RMSE=5.32  bias=-1.39  median|e|=0.0  max|e|=29


### 6. Model size and speed

In [21]:
ev.fuse()
PARAMS_M = sum(p.numel() for p in ev.model.parameters()) / 1e6
for im in imgs[:5]:                                   # warm-up
    ev.predict(os.path.join(img_dir, im), conf=CONF_PRIMARY, iou=IOU_NMS, max_det=MAX_DET, imgsz=IMGSZ, verbose=False, device=0)
runs = []
for _ in range(3):                                    # median of 3 passes, batch size 1, includes pre- and post-processing
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for im in imgs:
        ev.predict(os.path.join(img_dir, im), conf=CONF_PRIMARY, iou=IOU_NMS, max_det=MAX_DET, imgsz=IMGSZ, verbose=False, device=0)
    torch.cuda.synchronize()
    runs.append(len(imgs) / (time.perf_counter() - t0))
FPS = statistics.median(runs)
print(f"Parameters (fused): {PARAMS_M:.2f} M | FPS (median of 3): {FPS:.1f}  runs={[round(x,1) for x in runs]}")

Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
Parameters (fused): 3.13 M | FPS (median of 3): 103.8  runs=[101.7, 105.8, 103.8]


### 7. Save results

In [22]:
os.makedirs(RESULTS_DIR, exist_ok=True)
row = {"dataset": DATASET, "model": MODEL, "box": BOX, "seed": SEED, "init": INIT,
       "ultralytics": ultralytics.__version__, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
       "epochs_run": EPOCHS_RUN, "train_images": report["train"]["images"], "val_images": report["val"]["images"],
       "test_images": len(imgs), "max_gt_count_test": max(gts), "params_M_fused": round(PARAMS_M, 3), "FPS": round(FPS, 1)}
for k, v in val_m.items():  row[f"val_{k}"] = round(v, 4)
for k, v in test_m.items(): row[f"test_{k}"] = round(v, 4)
for t in THRESHOLDS:
    for k, v in count_res[t].items():
        row[f"{k}@{t}"] = round(v, 3)
for k, v in count_res[CONF_PRIMARY].items():          # main results at the primary threshold
    row[k] = round(v, 3)

with open(f"{RESULTS_DIR}/{RUN_NAME}_metrics.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(row.keys())); w.writeheader(); w.writerow(row)
with open(f"{RESULTS_DIR}/{RUN_NAME}_per_image.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(per_image[0].keys())); w.writeheader(); w.writerows(per_image)
shutil.copy(BEST, f"{RESULTS_DIR}/{RUN_NAME}_best.pt")
if (SAVE_DIR / "results.csv").exists():
    shutil.copy(SAVE_DIR / "results.csv", f"{RESULTS_DIR}/{RUN_NAME}_training_curve.csv")
print("Saved to", RESULTS_DIR)
for k in ["dataset", "model", "box", "init", "epochs_run", "test_P", "test_R", "test_F1", "test_mAP50", "test_mAP50_95", "MAE", "MAPE_pct", "RMSE", "bias", "FPS"]:
    print(f"  {k:14s} {row[k]}")

Saved to /kaggle/working/results
  dataset        public
  model          yolov8n
  box            HBB
  init           yolov8n.pt
  epochs_run     100
  test_P         0.9345
  test_R         0.8326
  test_F1        0.8806
  test_mAP50     0.8825
  test_mAP50_95  0.749
  MAE            2.0
  MAPE_pct       15.867
  RMSE           4.594
  bias           -0.257
  FPS            103.8
